# **Transfer Learning: Vision Transformers**
## **Car Make & Model Classification (Stanford Cars Dataset)**

---

Transfer learning is a technique where a pre-trained model – which has already learned features from one task – is used as the starting point for a similar task. This saves time and resources by leveraging the existing knowledge of the model instead of training a new model from scratch.

In this notebook, we apply transfer learning for **car make and model classification** using a Vision Transformer (ViT) fine-tuned on the **Stanford Cars Dataset** (196 classes of cars organized by make and model).

We use [google/vit-base-patch16-224](https://huggingface.co/google/vit-base-patch16-224) from the Hugging Face hub.

### Dataset structure expected on Kaggle:
```
car_data/
│── train/
│   ├── Acura Integra Type R 2001/
│   │   ├── image1.jpg
│   ├── BMW 3 Series 2012/
│   │   ├── image2.jpg
│── test/
│   ├── Acura Integra Type R 2001/
│   ├── BMW 3 Series 2012/
```

In [ ]:
!pip install evaluate transformers[torch] datasets gradio --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from datasets import DatasetDict, load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification
from transformers import Trainer, TrainingArguments
import evaluate

### (Optional) Login to Hugging Face Hub to push the trained model
---
Add your `HF_TOKEN` to your Kaggle secrets (Add-ons → Secrets) and run the cell below.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# -- OR: use Kaggle secrets --
# from kaggle_secrets import UserSecretsClient
# import os
# secrets = UserSecretsClient()
# os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])

### Load the Stanford Cars Dataset
---
The dataset from Kaggle (`jutrera/stanford-car-dataset-by-classes-folder`) is already split into `train/` and `test/` folders, with one sub-folder per class (196 classes in total). Each class folder name is the car's **make + model + year**, e.g. `Acura Integra Type R 2001`.

In [ ]:
import os
from datasets import load_dataset, DatasetDict

# --- Kaggle path (adjust if needed) ---
DATA_DIR = "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder"

# Load train and test splits from the folder structure
raw_dataset = load_dataset(
    "imagefolder",
    data_dir=DATA_DIR,
)

print(raw_dataset)
print(f"Number of classes: {raw_dataset['train'].features['label'].num_classes}")
print(f"Train images: {len(raw_dataset['train'])}")
print(f"Test  images: {len(raw_dataset['test'])}")

In [ ]:
# Class names = car make + model + year (e.g. "BMW 3 Series 2012")
label_names = raw_dataset['train'].features['label'].names
print(f"Total labels: {len(label_names)}")
print("First 10:", label_names[:10])

### Create train / validation / test splits
---
We split the original `train` set 80 / 20 into **train** and **validation**.
The original `test` split is kept as the final hidden test set.

In [ ]:
split = raw_dataset['train'].train_test_split(test_size=0.2, seed=42)

our_dataset = DatasetDict({
    'train':      split['train'],
    'validation': split['test'],
    'test':       raw_dataset['test'],
})

print(our_dataset)

In [ ]:
# Label <-> ID mappings
labels   = label_names
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

print(f"{len(labels)} classes")

### Visualise some samples
---

In [ ]:
def show_samples(ds, rows=3, cols=5):
    samples = ds.shuffle(seed=42).select(range(rows * cols))
    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for i in range(rows * cols):
        img   = samples[i]['image']
        label = samples[i]['label']
        ax = fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        # Wrap long class names for readability
        title = label_names[label].replace(' ', '\n', 2)
        plt.title(title, fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(our_dataset['train'])

## Preprocessing & Data Augmentation
---
We use `AutoImageProcessor` to apply the exact transforms expected by the ViT model (resize to 224×224, normalise).

Additionally, we apply **data augmentation** on the training split to improve generalisation. The Stanford Cars dataset has only ~80 images per class on average, which makes the model prone to overfitting. The following augmentations address this:

| Augmentation | Rationale |
|---|---|
| `RandomHorizontalFlip` | Cars can appear from both sides; the class label is direction-invariant |
| `ColorJitter` (brightness, contrast, saturation) | Simulates different lighting and weather conditions |
| `RandomRotation(±10°)` | Handles slight camera tilt; kept small to avoid unrealistic angles |

Augmentation is applied **only to the training split**. Validation and test images are only resized and normalised to ensure consistent evaluation.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T

MODEL_CHECKPOINT = 'google/vit-base-patch16-224'
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
processor

In [ ]:
# Augmentation pipeline applied only during training
train_augmentation = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(degrees=10),
])

def train_transforms(batch):
    images = [train_augmentation(img.convert('RGB')) for img in batch['image']]
    inputs = processor(images, return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

def eval_transforms(batch):
    """No augmentation for validation and test — only resize + normalise."""
    images = [img.convert('RGB') for img in batch['image']]
    inputs = processor(images, return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

# Alias used by show_predictions and predict_car below
transforms = eval_transforms

processed_dataset = our_dataset.with_transform(train_transforms)
# Override validation and test splits to use eval transforms (no augmentation)
processed_dataset['validation'] = our_dataset['validation'].with_transform(eval_transforms)
processed_dataset['test']       = our_dataset['test'].with_transform(eval_transforms)

In [ ]:
# Visualise augmentation effect on a single training image
sample_img = our_dataset['train'][0]['image'].convert('RGB')
fig, axes  = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(sample_img)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')
for i in range(1, 5):
    axes[i].imshow(train_augmentation(sample_img))
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Data augmentation examples (same image, different random transforms)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Data collation: stack pixel_values into a batch tensor
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch]),
    }

## Metrics
---
We track **accuracy** during training, and also report **top-5 accuracy** since we have 196 classes.

In [ ]:
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)

    # Top-5 accuracy
    top5_preds = np.argsort(logits, axis=1)[:, -5:]
    top5_correct = sum(labels[i] in top5_preds[i] for i in range(len(labels)))
    acc['top5_accuracy'] = top5_correct / len(labels)

    return acc

## Load Pre-trained ViT Model
---
We replace the classification head with one that outputs **196 logits** (one per car class).

In [ ]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

In [ ]:
# Freeze the backbone – only train the new classifier head
for name, p in model.named_parameters():
    if not name.startswith('classifier'):
        p.requires_grad = False

num_params       = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{num_params = :,} | {trainable_params = :,}")

## Training
---

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "./car-make-model-vit"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    num_train_epochs=5,
    learning_rate=3e-4,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,   # set True + provide HF_TOKEN to push to the Hub
    report_to='tensorboard',
    fp16=torch.cuda.is_available(),  # use mixed precision if GPU available
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed_dataset['train'],
    eval_dataset=processed_dataset['validation'],
    processing_class=processor,
)

In [ ]:
trainer.train()

### Evaluate on the held-out test set
---

In [ ]:
metrics = trainer.evaluate(processed_dataset['test'])
print(metrics)

### Visualise predictions
---

In [ ]:
def show_predictions(rows=3, cols=3):
    samples = our_dataset['test'].shuffle(seed=42).select(range(rows * cols))
    processed_samples = samples.with_transform(transforms)
    predictions = trainer.predict(processed_samples).predictions.argmax(axis=1)

    fig = plt.figure(figsize=(cols * 5, rows * 5))
    for i in range(rows * cols):
        img        = samples[i]['image']
        true_label = id2label[samples[i]['label']]
        pred_label = id2label[predictions[i]]
        color      = 'green' if true_label == pred_label else 'red'
        title      = f"True: {true_label}\nPred: {pred_label}"
        ax = fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(title, fontsize=8, color=color)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_predictions()

### Confusion matrix – top confused class pairs
---
With 196 classes a full confusion matrix is unreadable. Instead we extract the **top-15 most confused class pairs** — these reveal where the model struggles most (typically same make across model years, or visually similar body styles).

In [ ]:
from collections import Counter

# Collect all test predictions
test_output     = trainer.predict(processed_dataset['test'])
test_preds      = test_output.predictions.argmax(axis=1)
test_true       = test_output.label_ids

# Find wrong predictions and count (true, predicted) pairs
wrong_pairs = [
    (id2label[true], id2label[pred])
    for true, pred in zip(test_true, test_preds)
    if true != pred
]
top_confused = Counter(wrong_pairs).most_common(15)

print(f"Total test errors: {len(wrong_pairs)} / {len(test_true)}")
print(f"\nTop 15 confused class pairs (true → predicted):")
for (true_cls, pred_cls), count in top_confused:
    print(f"  {count:3d}×  '{true_cls}'  →  '{pred_cls}'")

In [ ]:
# Bar chart of top confused pairs
pairs_labels = [f"{t.split(' ')[-1]}\n→ {p.split(' ')[-1]}" for (t, p), _ in top_confused]
counts       = [c for _, c in top_confused]

fig, ax = plt.subplots(figsize=(14, 5))
ax.barh(pairs_labels[::-1], counts[::-1], color='steelblue')
ax.set_xlabel('Number of misclassifications')
ax.set_title('Top 15 confused class pairs on the test set')
plt.tight_layout()
plt.show()

### Failure analysis & model limitations
---
We visualise misclassified images and discuss the underlying reasons.

**Known limitations of the fine-tuned ViT on Stanford Cars:**

- **Year confusion** — The model frequently confuses the same make/model across consecutive model years (e.g. BMW 3 Series 2011 vs 2012). Visual differences between years are subtle (minor grille changes, headlight shape), and the dataset provides no metadata beyond the folder name.
- **Rare classes** — Classes with fewer training images (some classes have < 50 images) show higher error rates. The model has not seen enough variation in lighting, angle, and background.
- **Viewpoint sensitivity** — Rear-facing or side-profile images are harder than front-facing ones, as ViT patch embeddings are sensitive to the spatial distribution of features.
- **Background clutter** — Images taken in parking lots or at car shows with other vehicles in the background occasionally mislead the model.
- **Fine-grained discrimination** — Distinguishing between closely related trims of the same model (e.g. sedan vs coupe of the same generation) is difficult without explicit trim-level annotations.

In [ ]:
# Show 6 misclassified examples with true and predicted labels
wrong_indices = [
    i for i, (true, pred) in enumerate(zip(test_true, test_preds))
    if true != pred
][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, idx in zip(axes.flat, wrong_indices):
    img        = our_dataset['test'][idx]['image']
    true_lbl   = id2label[test_true[idx]]
    pred_lbl   = id2label[test_preds[idx]]
    ax.imshow(img)
    ax.set_title(f"True:  {true_lbl}\nPred: {pred_lbl}", fontsize=7, color='red')
    ax.axis('off')

plt.suptitle('Failure cases – misclassified test images', fontsize=11)
plt.tight_layout()
plt.show()

### Save the model
---

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

In [ ]:
# Optional: push to Hugging Face Hub
# trainer.push_to_hub()

### Predict a single image
---

In [ ]:
from PIL import Image
from transformers import AutoImageProcessor, ViTForImageClassification

def predict_car(image_path: str, model_dir: str = OUTPUT_DIR, top_k: int = 5):
    """
    Predict the make and model of a car from an image.
    Returns the top-k predictions with confidence scores.
    """
    proc = AutoImageProcessor.from_pretrained(model_dir)
    mdl  = ViTForImageClassification.from_pretrained(model_dir)
    mdl.eval()

    image  = Image.open(image_path).convert('RGB')
    inputs = proc(image, return_tensors='pt')

    with torch.no_grad():
        outputs = mdl(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0]

    top_k_values, top_k_indices = torch.topk(probs, top_k)

    results = [
        {"label": mdl.config.id2label[idx.item()], "confidence": f"{val.item():.1%}"}
        for idx, val in zip(top_k_indices, top_k_values)
    ]

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"{results[0]['label']}\n({results[0]['confidence']})", fontsize=12)
    plt.axis('off')
    plt.show()

    return results

# Example:
# predict_car("/path/to/car.jpg")

### Quick Gradio demo
---

In [ ]:
import gradio as gr
from transformers import pipeline

classifier = pipeline('image-classification', model=OUTPUT_DIR)

def classify_car(image):
    results = classifier(image, top_k=5)
    return {r['label']: r['score'] for r in results}

iface = gr.Interface(
    fn=classify_car,
    inputs=gr.Image(type='filepath'),
    outputs=gr.Label(num_top_classes=5),
    title='Car Make & Model Classification (ViT Transfer Learning)',
    description='Upload a car photo and the model will predict the make, model, and year.',
)

iface.launch()